# Notebook 13 — Bayesian A/B Testing

## Background

A/B testing is the most common statistical task in tech companies. You have a control and a treatment, you run an experiment, and you want to know: is the treatment better, worse, or equivalent to the control?

The frequentist approach uses a null hypothesis test: compute p(data | H0), and if it's small enough, reject H0 in favor of H1. This works but has awkward properties:
- You can't say 'there is a 90% chance the treatment is better' — only 'we reject H0 at significance level alpha'
- Optional stopping is invalid — peeking at results before the sample size is met inflates false positive rates
- Effect size uncertainty is expressed through confidence intervals, but interpreting them correctly is notoriously difficult

The Bayesian approach computes the posterior distribution of the treatment effect directly:
- `P(theta_B > theta_A | data)` — probability treatment is better
- `P(|theta_B - theta_A| > ROPE | data)` — probability the effect is practically meaningful
- Full posterior uncertainty over effect size
- Natural stopping rule: stop when posterior is conclusive (HDI vs ROPE)

## What You'll Learn

- Full Bayesian A/B test: Beta-Binomial model end-to-end
- Comparing probabilities, lift, and expected revenue impact
- Why frequentist p-values and Bayesian posteriors often disagree
- The optional stopping problem and how Bayesian methods handle it
- Multi-armed bandit as a natural Bayesian generalization
- Business impact framing: translating posterior into expected value

## Why This Matters

Every product experiment, marketing campaign test, and clinical trial is an A/B test at heart. Getting the statistical framework right determines whether you make decisions based on genuine evidence or noise. Bayesian A/B testing has grown substantially in industry (Google, Microsoft, Netflix all use Bayesian frameworks for some experiments) because it answers the questions practitioners actually ask.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — Frequentist vs. Bayesian A/B Test: Side by Side

We'll run both approaches on the same data so you can see exactly what each tells you.

In [ ]:
np.random.seed(42)

# Experiment: new signup flow design
# True rates: control 8%, treatment 10%
true_p_ctrl = 0.08
true_p_trt  = 0.10

n_ctrl, n_trt = 2000, 2000
k_ctrl = np.random.binomial(n_ctrl, true_p_ctrl)
k_trt  = np.random.binomial(n_trt,  true_p_trt)

print(f'Experiment: Signup Flow A/B Test')
print(f'  Control:   {k_ctrl}/{n_ctrl} = {k_ctrl/n_ctrl:.2%}')
print(f'  Treatment: {k_trt}/{n_trt} = {k_trt/n_trt:.2%}')
print(f'  True rates: {true_p_ctrl:.1%} (ctrl), {true_p_trt:.1%} (trt)')
print()

# ── Frequentist: two-proportion z-test ───────────────────────────────────────
from scipy.stats import chi2_contingency
table = np.array([[k_ctrl, n_ctrl - k_ctrl],
                   [k_trt,  n_trt  - k_trt]])
chi2, p_value, dof, expected = chi2_contingency(table, correction=False)

print('Frequentist (chi-squared test):')
print(f'  chi2 = {chi2:.3f}, p-value = {p_value:.4f}')
print(f'  Decision (alpha=0.05): {"Reject H0" if p_value < 0.05 else "Fail to reject H0"}')
print('  Interpretation: "Data is unlikely under H0 (equal rates)"')
print('  What you CANNOT say: "There is a 95% chance treatment is better"')
print()

# ── Bayesian: Beta-Binomial model ────────────────────────────────────────────
alpha_prior, beta_prior = 1, 1
post_ctrl = stats.beta(alpha_prior + k_ctrl, beta_prior + n_ctrl - k_ctrl)
post_trt  = stats.beta(alpha_prior + k_trt,  beta_prior + n_trt  - k_trt)

n_mc = 200_000
samp_ctrl = post_ctrl.rvs(n_mc)
samp_trt  = post_trt.rvs(n_mc)
diff_samp = samp_trt - samp_ctrl
lift_samp = (samp_trt - samp_ctrl) / samp_ctrl

p_trt_better = np.mean(samp_trt > samp_ctrl)
p_lift_5pct  = np.mean(lift_samp > 0.05)

print('Bayesian (Beta-Binomial):')
print(f'  P(treatment > control): {p_trt_better:.4f}')
print(f'  P(lift > 5%):           {p_lift_5pct:.4f}')
print(f'  Posterior mean difference: {diff_samp.mean()*100:.2f}%')
print(f'  94% HDI on difference: [{np.percentile(diff_samp, 3)*100:.2f}%, {np.percentile(diff_samp, 97)*100:.2f}%]')
print('  Interpretation: direct probability statements about the effect')

In [ ]:
# Visualize the comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Marginal posteriors
ax = axes[0]
x_range = np.linspace(0.04, 0.16, 300)
ax.fill_between(x_range, post_ctrl.pdf(x_range), alpha=0.4, color='steelblue', label=f'Control (n={n_ctrl}, k={k_ctrl})')
ax.fill_between(x_range, post_trt.pdf(x_range), alpha=0.4, color='darkorange', label=f'Treatment (n={n_trt}, k={k_trt})')
ax.plot(x_range, post_ctrl.pdf(x_range), 'steelblue', lw=2)
ax.plot(x_range, post_trt.pdf(x_range), 'darkorange', lw=2)
ax.axvline(true_p_ctrl, color='steelblue', linestyle=':', lw=1.5, alpha=0.8)
ax.axvline(true_p_trt,  color='darkorange', linestyle=':', lw=1.5, alpha=0.8)
ax.set_xlabel('Conversion Rate'); ax.set_ylabel('Density')
ax.set_title('Posterior Distributions\nDotted = true rates')
ax.legend(fontsize=8)

# Difference posterior
ax = axes[1]
xd = np.linspace(-0.03, 0.08, 300)
kde_diff = stats.gaussian_kde(diff_samp)
ax.fill_between(xd, kde_diff(xd), alpha=0.4, color='seagreen')
ax.plot(xd, kde_diff(xd), 'seagreen', lw=2)
ax.axvline(0, color='black', linestyle='--', lw=1.5, label='No difference')
ax.axvline(diff_samp.mean(), color='red', linestyle='--', lw=1.5,
           label=f'Mean: {diff_samp.mean()*100:.2f}%')
ax.fill_between([np.percentile(diff_samp, 3), np.percentile(diff_samp, 97)],
                [0, 0], [kde_diff(np.percentile(diff_samp, [3, 97]))[0], kde_diff(np.percentile(diff_samp, [3, 97]))[1]],
                alpha=0.15, color='red')
ax.set_xlabel('Treatment - Control'); ax.set_ylabel('Density')
ax.set_title(f'Posterior Difference\nP(trt > ctrl) = {p_trt_better:.3f}')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))

# Lift posterior
ax = axes[2]
lift_clip = lift_samp[(lift_samp > -0.5) & (lift_samp < 1.5)]
xr_lift = np.linspace(-0.3, 1.0, 300)
kde_lift = stats.gaussian_kde(lift_clip)
ax.fill_between(xr_lift, kde_lift(xr_lift), alpha=0.4, color='purple')
ax.plot(xr_lift, kde_lift(xr_lift), 'purple', lw=2)
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.axvline(0.05, color='red', linestyle='--', lw=1.5, label='5% lift threshold')
ax.set_xlabel('Relative Lift'); ax.set_ylabel('Density')
ax.set_title(f'Posterior Lift\nP(lift > 5%) = {p_lift_5pct:.3f}')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))

plt.suptitle('Bayesian A/B Test: Signup Flow Conversion Rate', fontsize=12)
plt.tight_layout()
plt.show()

## Part 2 — Business Impact: Expected Value Framing

Stakeholders rarely care about p-values or posterior probabilities — they care about revenue, users, or cost. Bayesian posteriors make it easy to compute expected business impact and its uncertainty.

In [ ]:
# Business impact: monthly expected signups and revenue
monthly_visitors = 100_000
revenue_per_signup = 45.0  # $45 per new user (LTV estimate)

# Current (control) signups per month
current_signups = samp_ctrl * monthly_visitors
new_signups     = samp_trt  * monthly_visitors
incremental     = (samp_trt - samp_ctrl) * monthly_visitors
incremental_rev = incremental * revenue_per_signup

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(incremental, bins=80, color='seagreen', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.axvline(incremental.mean(), color='red', lw=2, label=f'Mean: {incremental.mean():.0f} signups')
lo, hi = np.percentile(incremental, [3, 97])
ax.axvspan(lo, hi, alpha=0.15, color='red', label=f'94% CI: [{lo:.0f}, {hi:.0f}]')
ax.set_xlabel('Incremental Signups/Month')
ax.set_ylabel('Density')
ax.set_title('Monthly Incremental Signups\nIf we deploy treatment')
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(incremental_rev / 1000, bins=80, color='gold', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.axvline(incremental_rev.mean() / 1000, color='red', lw=2,
           label=f'Mean: ${incremental_rev.mean()/1000:.0f}k/month')
lo_r, hi_r = np.percentile(incremental_rev / 1000, [3, 97])
ax.axvspan(lo_r, hi_r, alpha=0.15, color='red',
           label=f'94% CI: [${lo_r:.0f}k, ${hi_r:.0f}k]')
ax.set_xlabel('Incremental Revenue ($k/month)')
ax.set_ylabel('Density')
ax.set_title('Monthly Incremental Revenue\nLTV = $45/signup')
ax.legend(fontsize=8)

plt.suptitle('Business Impact Framing: Posterior -> Expected Value', fontsize=12)
plt.tight_layout()
plt.show()

print('Business Impact Summary:')
print(f'  Expected incremental signups/month: {incremental.mean():.0f}')
print(f'  Expected incremental revenue/month: ${incremental_rev.mean():,.0f}')
print(f'  P(positive impact):                 {np.mean(incremental > 0):.3f}')
print(f'  P(revenue > $10k/month):            {np.mean(incremental_rev > 10_000):.3f}')

## Part 3 — The Optional Stopping Problem

**Frequentist testing and optional stopping don't mix**. If you peek at your p-value periodically and stop the experiment when p < 0.05, your actual false positive rate is much higher than 5% — because you've run multiple tests (one at each look), and each look has a chance of finding a spurious significant result.

Bayesian testing is naturally sequential. The posterior at any point in time represents your current state of belief, updated correctly by all data seen so far. You can peek at the posterior as often as you want — it updates correctly each time.

In [ ]:
# Simulate: frequentist vs Bayesian under sequential observation
np.random.seed(42)

# True effect: none (H0 is true)
true_p_null_ctrl = 0.10
true_p_null_trt  = 0.10  # same as control

n_total = 3000
ctrl_obs = np.random.binomial(1, true_p_null_ctrl, n_total)
trt_obs  = np.random.binomial(1, true_p_null_trt,  n_total)

# Track metrics at each time step
check_points = range(100, n_total + 1, 50)
p_values_seq       = []
p_trt_better_seq   = []
diff_mean_seq      = []
diff_ci_lo_seq     = []
diff_ci_hi_seq     = []

for n in check_points:
    k_c = ctrl_obs[:n].sum()
    k_t = trt_obs[:n].sum()
    
    # Frequentist p-value
    table = np.array([[k_c, n - k_c], [k_t, n - k_t]])
    try:
        from scipy.stats import chi2_contingency
        _, p, _, _ = chi2_contingency(table, correction=False)
    except:
        p = 1.0
    p_values_seq.append(p)
    
    # Bayesian posterior
    sc = stats.beta(1 + k_c, 1 + n - k_c).rvs(20000)
    st = stats.beta(1 + k_t, 1 + n - k_t).rvs(20000)
    d  = st - sc
    p_trt_better_seq.append(np.mean(d > 0))
    diff_mean_seq.append(d.mean())
    diff_ci_lo_seq.append(np.percentile(d, 3))
    diff_ci_hi_seq.append(np.percentile(d, 97))

check_points = list(check_points)
p_values_seq = np.array(p_values_seq)
diff_ci_lo_seq = np.array(diff_ci_lo_seq)
diff_ci_hi_seq = np.array(diff_ci_hi_seq)
diff_mean_seq  = np.array(diff_mean_seq)

# Count: how often does p < 0.05 (false positives from peeking)?
n_false_pos_freq = (p_values_seq < 0.05).sum()
n_looks = len(p_values_seq)

# Bayesian: how often does 94% HDI exclude 0?
bayesian_conclusive = ((diff_ci_lo_seq > 0) | (diff_ci_hi_seq < 0)).sum()

print(f'Sequential testing (H0 is TRUE: no effect)')
print(f'  Frequentist: p < 0.05 on {n_false_pos_freq}/{n_looks} looks ({n_false_pos_freq/n_looks:.1%}) -- inflated FPR!')
print(f'  Bayesian: HDI excludes 0 on {bayesian_conclusive}/{n_looks} looks ({bayesian_conclusive/n_looks:.1%})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.plot(check_points, p_values_seq, 'darkorange', lw=1.5, label='Frequentist p-value')
ax.axhline(0.05, color='red', linestyle='--', lw=1.5, label='alpha = 0.05')
ax.fill_between(check_points, 0, 0.05, alpha=0.1, color='red')
# Mark spurious significant results
spurious = p_values_seq < 0.05
ax.scatter(np.array(check_points)[spurious], p_values_seq[spurious],
           color='red', s=40, zorder=5, label=f'"Significant" ({spurious.sum()} spurious)')
ax.set_ylabel('p-value')
ax.set_title('Frequentist: Optional Stopping Inflates False Positive Rate\n'
             'p < 0.05 sometimes even though H0 is true!')
ax.legend(fontsize=8)
ax.set_ylim(0, 0.3)

ax = axes[1]
ax.plot(check_points, np.array(p_trt_better_seq), 'steelblue', lw=1.5, label='P(trt > ctrl)')
ax.fill_between(check_points, diff_ci_lo_seq * 100, diff_ci_hi_seq * 100,
                alpha=0.2, color='steelblue', label='94% HDI on difference (%)')
ax.axhline(0.5, color='black', linestyle='--', lw=1, alpha=0.7)
ax.axhline(0, color='gray', linestyle='-', lw=0.5)
ax.set_xlabel('n per arm')
ax.set_ylabel('P(trt > ctrl) / HDI')
ax.set_title('Bayesian: Posterior stays near 0.5 throughout -- correctly reflects no effect')
ax.legend(fontsize=8)

plt.suptitle('Sequential Testing: Frequentist vs. Bayesian under H0 (No Effect)', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — Multi-Armed Bandit: Thompson Sampling

A/B testing treats exploration and exploitation as separate phases: first run the experiment (exploration), then deploy the winner (exploitation). But you could continuously update beliefs and steer traffic toward better-performing variants in real time.

**Thompson Sampling** is a Bayesian bandit algorithm:
1. Maintain a Beta posterior over each arm's success rate
2. At each decision: sample from each arm's posterior; serve the arm whose sample was highest
3. Observe the outcome; update that arm's posterior

This naturally balances exploration (uncertain arms get sampled more when their posterior is wide) and exploitation (the currently best arm gets chosen more often).

In [ ]:
np.random.seed(42)

# Thompson Sampling: 3-armed bandit
# True rates: 0.08, 0.10, 0.12 -- we don't know these
true_rates = np.array([0.08, 0.10, 0.12])
n_arms = len(true_rates)

# Priors: Beta(1, 1) = uniform for each arm
alphas = np.ones(n_arms)
betas  = np.ones(n_arms)

n_rounds = 5000
rewards  = np.zeros(n_rounds)
choices  = np.zeros(n_rounds, dtype=int)
cumulative_traffic = np.zeros((n_rounds, n_arms))

for t in range(n_rounds):
    # Sample from each arm's posterior
    theta_samples = np.array([
        stats.beta(alphas[i], betas[i]).rvs()
        for i in range(n_arms)
    ])
    
    # Choose the arm with the highest sample
    chosen = np.argmax(theta_samples)
    choices[t] = chosen
    
    # Observe reward
    reward = np.random.binomial(1, true_rates[chosen])
    rewards[t] = reward
    
    # Update posterior
    alphas[chosen] += reward
    betas[chosen]  += 1 - reward
    
    # Track cumulative traffic
    if t > 0:
        cumulative_traffic[t] = cumulative_traffic[t-1]
    cumulative_traffic[t, chosen] += 1

# Traffic share over time
traffic_share = cumulative_traffic / (np.arange(1, n_rounds + 1)[:, None])

print('Thompson Sampling: 3-Armed Bandit')
print(f'True rates: {true_rates}')
print(f'Final posteriors (alpha, beta):')
for i in range(n_arms):
    post_mean = alphas[i] / (alphas[i] + betas[i])
    traffic = cumulative_traffic[-1, i]
    print(f'  Arm {i} (true={true_rates[i]:.2f}): post_mean={post_mean:.3f}, traffic={traffic:.0f} ({traffic/n_rounds:.1%})')

optimal_rate = true_rates.max()
actual_rate  = rewards.mean()
static_ab_rate = true_rates.mean()  # if we split equally
print(f'\nPerformance:')
print(f'  Optimal rate:    {optimal_rate:.4f}')
print(f'  Thompson actual: {actual_rate:.4f}')
print(f'  Static 1/3 split: {static_ab_rate:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Traffic allocation over time
ax = axes[0]
colors = ['steelblue', 'darkorange', 'seagreen']
for i in range(n_arms):
    ax.plot(traffic_share[:, i], color=colors[i], lw=2,
            label=f'Arm {i} (true={true_rates[i]:.2f})')
ax.axhline(1/3, color='gray', linestyle='--', lw=1, alpha=0.7, label='Equal split (1/3)')
ax.set_xlabel('Round'); ax.set_ylabel('Traffic Share')
ax.set_title('Thompson Sampling: Traffic Allocation\nConverges to best arm')
ax.legend(fontsize=8)

# Cumulative regret: how much reward did we miss vs. always choosing the best arm?
ax = axes[1]
cumulative_reward = rewards.cumsum()
optimal_cumulative = true_rates.max() * (np.arange(1, n_rounds + 1))
regret = optimal_cumulative - cumulative_reward

# Compare to random allocation
random_cumulative = true_rates.mean() * (np.arange(1, n_rounds + 1))
random_regret = optimal_cumulative - random_cumulative

ax.plot(regret, 'steelblue', lw=2, label='Thompson Sampling regret')
ax.plot(random_regret, 'darkorange', linestyle='--', lw=2, label='Random/equal regret')
ax.set_xlabel('Round'); ax.set_ylabel('Cumulative Regret')
ax.set_title('Cumulative Regret: Thompson vs. Equal Split\nLower = better')
ax.legend(fontsize=9)

plt.suptitle('Thompson Sampling: Bayesian Multi-Armed Bandit', fontsize=12)
plt.tight_layout()
plt.show()

## Part 5 — Complete A/B Test Workflow

Putting it all together: a repeatable Bayesian A/B test workflow.

In [ ]:
def bayesian_ab_test(
    n_ctrl, k_ctrl, n_trt, k_trt,
    rope=(-0.005, 0.005),
    prior=(1, 1),
    n_samples=200_000,
    revenue_per_conversion=None,
    monthly_visitors=None
):
    """
    Complete Bayesian A/B test analysis.
    
    Parameters:
    - n_ctrl, k_ctrl: sample size and conversions for control
    - n_trt, k_trt: sample size and conversions for treatment
    - rope: (lo, hi) region of practical equivalence
    - prior: (alpha, beta) for Beta prior
    - revenue_per_conversion: optional, for business impact
    - monthly_visitors: optional, for business impact
    
    Returns dict with all key metrics.
    """
    alpha_p, beta_p = prior
    post_ctrl = stats.beta(alpha_p + k_ctrl, beta_p + n_ctrl - k_ctrl)
    post_trt  = stats.beta(alpha_p + k_trt,  beta_p + n_trt  - k_trt)
    
    samp_ctrl = post_ctrl.rvs(n_samples)
    samp_trt  = post_trt.rvs(n_samples)
    diff      = samp_trt - samp_ctrl
    
    # Core metrics
    result = {
        'p_ctrl':           k_ctrl / n_ctrl,
        'p_trt':            k_trt  / n_trt,
        'post_mean_ctrl':   samp_ctrl.mean(),
        'post_mean_trt':    samp_trt.mean(),
        'p_trt_better':     float(np.mean(diff > 0)),
        'diff_mean':        float(diff.mean()),
        'diff_hdi_94':      (float(np.percentile(diff, 3)), float(np.percentile(diff, 97))),
        'p_inside_rope':    float(np.mean((diff >= rope[0]) & (diff <= rope[1]))),
    }
    
    # ROPE decision
    hdi_lo, hdi_hi = result['diff_hdi_94']
    if hdi_lo >= rope[0] and hdi_hi <= rope[1]:
        result['decision'] = 'equivalent'
    elif hdi_hi <= rope[0] or hdi_lo >= rope[1]:
        result['decision'] = 'meaningful_difference'
    else:
        result['decision'] = 'inconclusive'
    
    # Business impact (optional)
    if revenue_per_conversion and monthly_visitors:
        incremental = diff * monthly_visitors
        result['expected_monthly_revenue'] = float(incremental.mean() * revenue_per_conversion)
        result['revenue_ci_94'] = (
            float(np.percentile(incremental, 3) * revenue_per_conversion),
            float(np.percentile(incremental, 97) * revenue_per_conversion)
        )
        result['p_positive_revenue'] = float(np.mean(incremental > 0))
    
    return result

# Run on our signup flow experiment
result = bayesian_ab_test(
    n_ctrl=n_ctrl, k_ctrl=k_ctrl,
    n_trt=n_trt,   k_trt=k_trt,
    rope=(-0.005, 0.005),
    revenue_per_conversion=45,
    monthly_visitors=100_000
)

print('Bayesian A/B Test Report')
print('=' * 55)
print(f'  Control rate:            {result["p_ctrl"]:.2%}')
print(f'  Treatment rate:          {result["p_trt"]:.2%}')
print(f'  P(treatment > control):  {result["p_trt_better"]:.3f}')
print(f'  Posterior difference:    {result["diff_mean"]*100:.2f}%')
print(f'  94% HDI:                 [{result["diff_hdi_94"][0]*100:.2f}%, {result["diff_hdi_94"][1]*100:.2f}%]')
print(f'  P(inside ROPE +-0.5%):   {result["p_inside_rope"]:.3f}')
print(f'  Decision:                {result["decision"]}')
print()
if 'expected_monthly_revenue' in result:
    print(f'  Expected monthly revenue: ${result["expected_monthly_revenue"]:,.0f}')
    print(f'  Revenue 94% CI:          [${result["revenue_ci_94"][0]:,.0f}, ${result["revenue_ci_94"][1]:,.0f}]')
    print(f'  P(positive revenue):      {result["p_positive_revenue"]:.3f}')

## Summary

**Bayesian A/B Test Workflow**:
1. Specify priors: `Beta(1, 1)` for conversion rates (uniform on [0, 1])
2. Observe data: `k_ctrl, n_ctrl, k_trt, n_trt`
3. Compute posteriors: `Beta(1 + k, 1 + n - k)` per arm
4. Sample posterior difference via Monte Carlo
5. Compute: `P(trt > ctrl)`, HDI, ROPE decision
6. Translate to business impact: incremental revenue, signups

**Key advantages over frequentist A/B testing**:
- Direct probability statements: `P(trt > ctrl) = 0.94`
- Safe sequential monitoring: peek as often as needed
- ROPE for practical significance: separate statistical from practical relevance
- Business impact framing: posterior -> expected value with uncertainty

**Thompson Sampling**: use when you can steer traffic dynamically — it converges to optimal arm while minimizing regret, outperforming static A/B splits.

**Next**: Notebook 14 — Sequential/Adaptive Testing. Formal sequential stopping rules, error control, and adaptive sample size.